### GOLD TESTING - DIMENSION USERS (SCD2)

#### Purpose
- Validate `coffee.gold.dim_users` against `coffee.silver.users`
- Ensure SCD Type 2 logic is working correctly
- Verify current records using `__END_AT IS NULL`
- Validate referential integrity from `fact_transactions` to `dim_users`

#### Tests Covered
1. Silver vs Gold row count reconciliation (current rows only)
2. Null checks on business key (`user_id`)
3. SCD2 sanity check: only 1 current row per `user_id`
4. SCD2 sanity check: invalid date ranges (`__END_AT < __START_AT`)
5. Referential integrity: `fact_transactions.user_id` must exist in current `dim_users`


In [0]:
-- TEST 1: SILVER vs GOLD CURRENT ROW COUNT
SELECT
  'dim_users_count_recon' AS test_name,
  (SELECT COUNT(*) FROM coffee.silver.users) AS silver_count,
  (SELECT COUNT(*) FROM coffee.gold.dim_users WHERE __END_AT IS NULL) AS gold_current_count;

In [0]:
-- TEST 2: NULL CHECK ON BUSINESS KEY
SELECT
  'dim_users_null_user_id' AS test_name,
  COUNT(*) AS null_key_count
FROM coffee.gold.dim_users
WHERE user_id IS NULL;

In [0]:
-- TEST 3: SCD2 SANITY - ONLY 1 CURRENT ROW PER USER
SELECT
  'dim_users_multiple_current_rows' AS test_name,
  COUNT(*) AS keys_with_multiple_current
FROM (
  SELECT user_id
  FROM coffee.gold.dim_users
  WHERE __END_AT IS NULL
  GROUP BY user_id
  HAVING COUNT(*) > 1
);

In [0]:
-- TEST 4: SCD2 SANITY - INVALID DATE RANGE CHECK
SELECT
  'dim_users_invalid_date_ranges' AS test_name,
  COUNT(*) AS invalid_rows
FROM coffee.gold.dim_users
WHERE __END_AT IS NOT NULL AND __END_AT < __START_AT;

In [0]:
-- TEST 5: REFERENTIAL INTEGRITY - FACT TRANSACTIONS -> DIM USERS
SELECT
  'RI_fact_transactions_user_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transactions f
LEFT JOIN coffee.gold.dim_users d
  ON f.user_id = d.user_id AND d.__END_AT IS NULL
WHERE  f.user_id IS NOT NULL
and d.user_id IS NULL;